In [ ]:
# AUTO-GENERATED by tools/build_kernel.py - edit the parts, not this file.
# part: src/jax_resnet.py sha256=07d671a682e2d15d
# part: kernels/tpu-probe/main.py sha256=a69e5a85be9fda71

# ======== src/jax_resnet.py ========
# Flax (linen) ResNet-18/34 whose submodule names mirror torchvision, so ImageNet weights and
# trained weights convert 1:1 in both directions (see convert_tv_to_flax / flax_to_tv_state_dict).
from functools import partial
from typing import Any, Sequence

import numpy as np
import jax
import jax.numpy as jnp
import flax.linen as nn

ARCH = {"resnet18": [2, 2, 2, 2], "resnet34": [3, 4, 6, 3]}
WIDTHS = [64, 128, 256, 512]


class BasicBlock(nn.Module):
    features: int
    stride: int = 1
    has_down: bool = False
    dtype: Any = jnp.float32

    @nn.compact
    def __call__(self, x, train: bool):
        conv = partial(nn.Conv, use_bias=False, dtype=self.dtype, param_dtype=jnp.float32)
        norm = partial(nn.BatchNorm, use_running_average=not train, momentum=0.9, epsilon=1e-5,
                       dtype=self.dtype, param_dtype=jnp.float32)
        y = conv(self.features, (3, 3), (self.stride, self.stride), padding=((1, 1), (1, 1)), name="conv1")(x)
        y = norm(name="bn1")(y)
        y = nn.relu(y)
        y = conv(self.features, (3, 3), (1, 1), padding=((1, 1), (1, 1)), name="conv2")(y)
        y = norm(name="bn2")(y)
        if self.has_down:
            x = conv(self.features, (1, 1), (self.stride, self.stride), padding=((0, 0), (0, 0)), name="down_conv")(x)
            x = norm(name="down_bn")(x)
        return nn.relu(x + y)


class ResNetBackbone(nn.Module):
    arch: str = "resnet18"
    width_mult: float = 1.0
    dtype: Any = jnp.float32

    @nn.compact
    def __call__(self, x, train: bool):
        """x: [B,H,W,3] already normalised. Returns the final feature map [B,H/32,W/32,C]."""
        widths = [int(round(w * self.width_mult)) for w in WIDTHS]
        stem = widths[0]
        x = x.astype(self.dtype)
        x = nn.Conv(stem, (7, 7), (2, 2), padding=((3, 3), (3, 3)), use_bias=False, dtype=self.dtype,
                    param_dtype=jnp.float32, name="conv1")(x)
        x = nn.BatchNorm(use_running_average=not train, momentum=0.9, epsilon=1e-5, dtype=self.dtype,
                         param_dtype=jnp.float32, name="bn1")(x)
        x = nn.relu(x)
        x = nn.max_pool(x, (3, 3), (2, 2), padding=((1, 1), (1, 1)))
        in_f = stem
        for li, (nb, f) in enumerate(zip(ARCH[self.arch], widths)):
            for bi in range(nb):
                stride = 2 if (bi == 0 and li > 0) else 1
                has_down = (bi == 0) and (stride != 1 or in_f != f)
                x = BasicBlock(f, stride, has_down, self.dtype, name=f"layer{li + 1}_{bi}")(x, train)
                in_f = f
        return x


# ---------------------------------------------------------------- torchvision <-> flax weight conversion
def _conv_to_flax(w):  # [O,I,kh,kw] -> [kh,kw,I,O]
    return np.asarray(w).transpose(2, 3, 1, 0)


def _conv_to_tv(k):
    return np.asarray(k).transpose(3, 2, 0, 1)


def convert_tv_to_flax(sd, arch="resnet18"):
    """torchvision ResNet state_dict (tensors or numpy) -> (params, batch_stats) for ResNetBackbone (fc dropped)."""
    g = lambda k: np.asarray(sd[k].detach().cpu().numpy() if hasattr(sd[k], "detach") else sd[k])
    params, stats = {}, {}

    def bn(dst_p, dst_s, name, prefix):
        dst_p[name] = {"scale": g(prefix + ".weight"), "bias": g(prefix + ".bias")}
        dst_s[name] = {"mean": g(prefix + ".running_mean"), "var": g(prefix + ".running_var")}

    params["conv1"] = {"kernel": _conv_to_flax(g("conv1.weight"))}
    bn(params, stats, "bn1", "bn1")
    for li, nb in enumerate(ARCH[arch]):
        for bi in range(nb):
            tv = f"layer{li + 1}.{bi}"
            nm = f"layer{li + 1}_{bi}"
            p, s = {}, {}
            p["conv1"] = {"kernel": _conv_to_flax(g(tv + ".conv1.weight"))}
            bn(p, s, "bn1", tv + ".bn1")
            p["conv2"] = {"kernel": _conv_to_flax(g(tv + ".conv2.weight"))}
            bn(p, s, "bn2", tv + ".bn2")
            if (tv + ".downsample.0.weight") in sd:
                p["down_conv"] = {"kernel": _conv_to_flax(g(tv + ".downsample.0.weight"))}
                bn(p, s, "down_bn", tv + ".downsample.1")
            params[nm], stats[nm] = p, s
    return params, stats


def flax_to_tv_state_dict(params, stats, arch="resnet18"):
    """Inverse of convert_tv_to_flax: returns a dict of numpy arrays with torchvision names (no fc)."""
    sd = {}
    a = lambda x: np.asarray(x)

    def bn(prefix, p, s):
        sd[prefix + ".weight"] = a(p["scale"])
        sd[prefix + ".bias"] = a(p["bias"])
        sd[prefix + ".running_mean"] = a(s["mean"])
        sd[prefix + ".running_var"] = a(s["var"])
        sd[prefix + ".num_batches_tracked"] = np.array(0, dtype=np.int64)

    sd["conv1.weight"] = _conv_to_tv(a(params["conv1"]["kernel"]))
    bn("bn1", params["bn1"], stats["bn1"])
    for li, nb in enumerate(ARCH[arch]):
        for bi in range(nb):
            tv = f"layer{li + 1}.{bi}"
            nm = f"layer{li + 1}_{bi}"
            p, s = params[nm], stats[nm]
            sd[tv + ".conv1.weight"] = _conv_to_tv(a(p["conv1"]["kernel"]))
            bn(tv + ".bn1", p["bn1"], s["bn1"])
            sd[tv + ".conv2.weight"] = _conv_to_tv(a(p["conv2"]["kernel"]))
            bn(tv + ".bn2", p["bn2"], s["bn2"])
            if "down_conv" in p:
                sd[tv + ".downsample.0.weight"] = _conv_to_tv(a(p["down_conv"]["kernel"]))
                bn(tv + ".downsample.1", p["down_bn"], s["down_bn"])
    return sd

# ======== kernels/tpu-probe/main.py ========
# ---- bench-gauges-tpu-probe: TPU kernel, NO dataset, short. Checks JAX on the Kaggle TPU and ResNet-18 train-step throughput.
import os, sys, time, json, subprocess
import numpy as np
import optax
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

def sh(c, t=60):
    try:
        r = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=t)
        return (r.stdout + r.stderr).strip()
    except Exception as e:
        return f"ERR {e}"

T0 = time.time()
print("== host", flush=True)
print(sh("nproc; free -g | head -2; python -V; df -h /kaggle/working | tail -1"))
print(sh("pip list 2>/dev/null | grep -i -E '^(jax|jaxlib|libtpu|torch|torch-xla|torch_xla|flax|optax|numpy|opencv|timm|onnx)'"))
print("== jax", jax.__version__, flush=True)
devs = jax.devices()
print("devices", devs, "local", jax.local_device_count(), "process_count", jax.process_count(), "t=%.1f" % (time.time() - T0), flush=True)
print("platform", devs[0].platform, getattr(devs[0], "device_kind", "?"))
try:
    print("memory_stats", devs[0].memory_stats())
except Exception as e:
    print("memory_stats err", e)

n = len(devs)
mesh = Mesh(np.array(devs), ("data",))
shard_b = NamedSharding(mesh, P("data"))
repl = NamedSharding(mesh, P())

# matmul sanity
a = jnp.ones((8192, 8192), jnp.bfloat16)
f = jax.jit(lambda x: x @ x)
f(a).block_until_ready()
t = time.time()
for _ in range(10):
    r = f(a)
r.block_until_ready()
dt = (time.time() - t) / 10
print("single-chip bf16 matmul 8192^3: %.1f TFLOPs" % (2 * 8192 ** 3 / dt / 1e12), flush=True)

class Net(nn.Module):
    @nn.compact
    def __call__(self, x, train):
        f = ResNetBackbone("resnet18", dtype=jnp.bfloat16)(x, train)
        f = f.mean((1, 2))
        return nn.Dense(10, dtype=jnp.float32)(f.astype(jnp.float32))

net = Net()
B = 64 * n
x0 = jnp.zeros((B, 224, 224, 3), jnp.float32)
vars_ = net.init(jax.random.PRNGKey(0), x0[:2], True)
params, stats = vars_["params"], vars_["batch_stats"]
tx = optax.adamw(1e-3)
opt = tx.init(params)
params, stats, opt = jax.device_put((params, stats, opt), repl)

@jax.jit
def step(params, stats, opt, xb, yb):
    def loss_fn(p):
        out, upd = net.apply({"params": p, "batch_stats": stats}, (xb.astype(jnp.float32) / 255.0 - 0.45) / 0.225, True, mutable=["batch_stats"])
        loss = optax.softmax_cross_entropy_with_integer_labels(out, yb).mean()
        return loss, upd["batch_stats"]
    (loss, new_stats), g = jax.value_and_grad(loss_fn, has_aux=True)(params)
    updates, opt2 = tx.update(g, opt, params)
    return optax.apply_updates(params, updates), new_stats, opt2, loss

xb_np = np.random.randint(0, 255, (B, 224, 224, 3), dtype=np.uint8)
yb_np = np.random.randint(0, 10, (B,), dtype=np.int32)
t = time.time()
xb = jax.device_put(xb_np, shard_b); yb = jax.device_put(yb_np, shard_b)
xb.block_until_ready()
print("h2d %.1f MB in %.3fs" % (xb_np.nbytes / 1e6, time.time() - t), flush=True)
t = time.time()
params, stats, opt, loss = step(params, stats, opt, xb, yb)
loss.block_until_ready()
print("compile+first step %.1fs loss %.3f" % (time.time() - t, float(loss)), flush=True)
t = time.time()
for _ in range(30):
    params, stats, opt, loss = step(params, stats, opt, xb, yb)
loss.block_until_ready()
dt = (time.time() - t) / 30
print("resnet18 bf16 train step global batch %d: %.1f ms -> %.0f img/s (all %d chips)" % (B, dt * 1e3, B / dt, n), flush=True)
# with per-step host->device transfer
t = time.time()
for _ in range(10):
    xb = jax.device_put(xb_np, shard_b)
    params, stats, opt, loss = step(params, stats, opt, xb, yb)
loss.block_until_ready()
dt = (time.time() - t) / 10
print("with h2d each step: %.1f ms -> %.0f img/s" % (dt * 1e3, B / dt), flush=True)
print("total secs %.1f" % (time.time() - T0))
